In [11]:
"""
Stage 1: SAR Baseline Experiment
Establish baseline with Simple Algorithm for Recommendation
"""

from recommenders.utils.timer import Timer
from recommenders.utils.constants import (
    DEFAULT_USER_COL, DEFAULT_ITEM_COL, DEFAULT_RATING_COL,
    DEFAULT_TIMESTAMP_COL, DEFAULT_PREDICTION_COL, SEED
)
from recommenders.evaluation.python_evaluation import (
    map_at_k, ndcg_at_k, precision_at_k, recall_at_k
)
from recommenders.models.sar import SAR
from recommenders.datasets.python_splitters import python_stratified_split
from recommenders.datasets import movielens
from datetime import datetime
import pandas as pd
import numpy as np
import sys
import os
import logging
import warnings
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.ERROR)


# Add path for benchmark_utils if needed
current_path = os.path.join(os.getcwd(), "examples", "06_benchmarks")
sys.path.append(current_path)

try:
    from benchmark_utils import *
except ImportError:
    print("Warning: benchmark_utils not found, defining functions locally")


In [12]:
# Set random seed
np.random.seed(SEED)

# Configuration
DATA_SIZE = "1m"
RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"=== Stage 1: SAR Baseline Experiment ===")
print(f"Start time: {datetime.now()}")

# Load data
print(f"\nLoading MovieLens {DATA_SIZE} dataset...")
df = movielens.load_pandas_df(
    size=DATA_SIZE,
    header=[DEFAULT_USER_COL, DEFAULT_ITEM_COL,
            DEFAULT_RATING_COL, DEFAULT_TIMESTAMP_COL]
)
print(f"Dataset shape: {df.shape}")

# Data split
print("\nSplitting data (75/25)...")
df_train, df_test = python_stratified_split(
    df,
    ratio=0.75,
    min_rating=1,
    filter_by="item",
    col_user=DEFAULT_USER_COL,
    col_item=DEFAULT_ITEM_COL
)
print(f"Train shape: {df_train.shape}, Test shape: {df_test.shape}")


=== Stage 1: SAR Baseline Experiment ===
Start time: 2025-06-03 12:01:12.296902

Loading MovieLens 1m dataset...


100%|██████████| 5.78k/5.78k [00:00<00:00, 21.5kKB/s]


Dataset shape: (1000209, 4)

Splitting data (75/25)...
Train shape: (750261, 4), Test shape: (249948, 4)


In [13]:

# SAR parameters
sar_params = {
    "similarity_type": "jaccard",
    "time_decay_coefficient": 30,
    "time_now": None,
    "timedecay_formula": True,
    "col_user": DEFAULT_USER_COL,
    "col_item": DEFAULT_ITEM_COL,
    "col_rating": DEFAULT_RATING_COL,
    "col_timestamp": DEFAULT_TIMESTAMP_COL,
}


In [14]:

# Train SAR model
print("\nTraining SAR model...")
model = SAR(**sar_params)
model.set_index(df_train)

with Timer() as t:
    model.fit(df_train)
train_time = t.interval

print(f"Training completed in {train_time:.4f} seconds")


Training SAR model...
Training completed in 2.8195 seconds


In [15]:

# Make predictions
print("\nMaking top-k recommendations...")
k = 10
with Timer() as t:
    top_k_scores = model.recommend_k_items(
        df_test,
        top_k=k,
        remove_seen=True
    )
predict_time = t.interval

print(f"Prediction completed in {predict_time:.4f} seconds")



Making top-k recommendations...
Prediction completed in 2.6908 seconds


In [16]:

# Evaluate
print("\nEvaluating model performance...")
eval_results = {
    "Model": "SAR",
    "Data_Size": DATA_SIZE,
    "Train_Time": train_time,
    "Predict_Time": predict_time,
    "K": k,
    "MAP": map_at_k(df_test, top_k_scores, k=k, **{"col_user": DEFAULT_USER_COL, "col_item": DEFAULT_ITEM_COL, "col_rating": DEFAULT_RATING_COL, "col_prediction": DEFAULT_PREDICTION_COL}),
    "NDCG@K": ndcg_at_k(df_test, top_k_scores, k=k, **{"col_user": DEFAULT_USER_COL, "col_item": DEFAULT_ITEM_COL, "col_rating": DEFAULT_RATING_COL, "col_prediction": DEFAULT_PREDICTION_COL}),
    "Precision@K": precision_at_k(df_test, top_k_scores, k=k, **{"col_user": DEFAULT_USER_COL, "col_item": DEFAULT_ITEM_COL, "col_rating": DEFAULT_RATING_COL, "col_prediction": DEFAULT_PREDICTION_COL}),
    "Recall@K": recall_at_k(df_test, top_k_scores, k=k, **{"col_user": DEFAULT_USER_COL, "col_item": DEFAULT_ITEM_COL, "col_rating": DEFAULT_RATING_COL, "col_prediction": DEFAULT_PREDICTION_COL})
}

# Analyze recommendation diversity
print("\nAnalyzing recommendation patterns...")
# Item popularity in training set
item_popularity = df_train.groupby(
    DEFAULT_ITEM_COL).size().reset_index(name='popularity')
item_popularity = item_popularity.sort_values('popularity', ascending=False)

# Check popular items in recommendations
rec_items = top_k_scores[DEFAULT_ITEM_COL].value_counts().head(20)
top_20_popular = item_popularity.head(20)[DEFAULT_ITEM_COL].values

popular_in_recs = sum([item in top_20_popular for item in rec_items.index])
eval_results["Popular_Items_in_Top20_Recs"] = popular_in_recs
eval_results["Unique_Items_Recommended"] = top_k_scores[DEFAULT_ITEM_COL].nunique()
eval_results["Catalog_Coverage"] = top_k_scores[DEFAULT_ITEM_COL].nunique(
) / df_train[DEFAULT_ITEM_COL].nunique()

# Cold start analysis
print("\nAnalyzing cold start performance...")
user_item_counts = df_train.groupby(DEFAULT_USER_COL).size()
cold_users = user_item_counts[user_item_counts <= 5].index
cold_user_test = df_test[df_test[DEFAULT_USER_COL].isin(cold_users)]

if len(cold_user_test) > 0:
    cold_user_preds = top_k_scores[top_k_scores[DEFAULT_USER_COL].isin(
        cold_users)]
    eval_results["Cold_Start_Precision@K"] = precision_at_k(
        cold_user_test, cold_user_preds, k=k,
        **{"col_user": DEFAULT_USER_COL, "col_item": DEFAULT_ITEM_COL, "col_rating": DEFAULT_RATING_COL, "col_prediction": DEFAULT_PREDICTION_COL}
    )
else:
    eval_results["Cold_Start_Precision@K"] = np.nan



Evaluating model performance...

Analyzing recommendation patterns...

Analyzing cold start performance...


In [17]:

# Save results
results_df = pd.DataFrame([eval_results])
results_file = os.path.join(RESULTS_DIR, "stage1_sar_results.csv")
results_df.to_csv(results_file, index=False)
print(f"\nResults saved to {results_file}")

# Save sample recommendations for analysis
sample_users = df_test[DEFAULT_USER_COL].sample(n=5, random_state=SEED)
sample_recs = top_k_scores[top_k_scores[DEFAULT_USER_COL].isin(sample_users)]
sample_file = os.path.join(
    RESULTS_DIR, "stage1_sar_sample_recommendations.csv")
sample_recs.to_csv(sample_file, index=False)



Results saved to results/stage1_sar_results.csv


In [18]:

# Print summary
print("\n=== Stage 1 Results Summary ===")
for key, value in eval_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

print(f"\nIdentified Issues:")
print(f"- Cannot handle explicit rating prediction")
print(
    f"- High popularity bias: {popular_in_recs}/20 top recommendations are popular items")
print(f"- Limited catalog coverage: {eval_results['Catalog_Coverage']:.2%}")
print(
    f"- Cold start performance: {eval_results.get('Cold_Start_Precision@K', 'N/A')}")

print(f"\nEnd time: {datetime.now()}")


=== Stage 1 Results Summary ===
Model: SAR
Data_Size: 1m
Train_Time: 2.8195
Predict_Time: 2.6908
K: 10
MAP: 0.1882
NDCG@K: 0.3121
Precision@K: 0.2803
Recall@K: 0.1106
Popular_Items_in_Top20_Recs: 6
Unique_Items_Recommended: 1184
Catalog_Coverage: 0.3195
Cold_Start_Precision@K: nan

Identified Issues:
- Cannot handle explicit rating prediction
- High popularity bias: 6/20 top recommendations are popular items
- Limited catalog coverage: 31.95%
- Cold start performance: nan

End time: 2025-06-03 12:01:30.797801
